In [1]:
!pip install yfinance

     |████████████████████████████████| 5.5 MB 4.6 MB/s 
  Created wheel for yfinance: filename=yfinance-0.1.55-py2.py3-none-any.whl size=22616 sha256=b31836b3394ed617323160716466c2480b5b5983cac975eaebfbebdae044e16b
  Stored in directory: /root/.cache/pip/wheels/aa/8a/36/59ed4f6fbcb6100967618eeb0696046bf9777a41ac2ff1f9b9
  Created wheel for multitasking: filename=multitasking-0.0.9-py3-none-any.whl size=8366 sha256=41e8e49ed3c236de55553a0771a3baea2e14a45daee5cce18bb063407e6400e6
  Stored in directory: /root/.cache/pip/wheels/ae/25/47/4d68431a7ec1b6c4b5233365934b74c1d4e665bf5f968d363a
Successfully built yfinance multitasking
  Attempting uninstall: lxml
    Found existing installation: lxml 4.5.0
    Uninstalling lxml-4.5.0:
      Successfully uninstalled lxml-4.5.0
You should consider upgrading via the '/opt/conda/bin/python3.7 -m pip install --upgrade pip' command.


In [2]:
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime
import yfinance as yf

import ipywidgets as ipw
from IPython.display import HTML, display,Javascript,clear_output
from ipywidgets import interact, interactive, fixed, interact_manual
# https://www.youtube.com/watch?v=abXuRM0QVD0&ab_channel=Elliott


def plot_chart(stock_df):
    import plotly.express as px
    fig = go.Figure()

    fig.add_trace(go.Candlestick(x=stock_df.index,
                    open=stock_df['Open'],
                    high=stock_df['High'],
                    low=stock_df['Low'],
                    close=stock_df['Close'],name='Candlestick'))

    #fig = px.line(google_df, x=google_df.index, y="30moving average")

    fig.add_trace(go.Scatter(x=stock_df.index, y=stock_df["150moving average"],
                        line=dict(color='yellow', width=4),
                        name='150 moving average'))

    fig.layout.yaxis=dict(
           autorange = True,
           fixedrange= False
       )


    fig.update_layout(
        title=stock_df.name+" Stock Price",
        title_x=0.5,
        xaxis_title="Date",
        yaxis_title="Stock Price",
        legend_title="Legend",
        font=dict(
            family="Courier New, monospace",
            size=12,
            color="RebeccaPurple"
        )
    )

    fig.update_layout(legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ))

    fig.update_xaxes(title_font_family="Arial")
    fig.show()
    
def linechart(stock_df):
    fig = go.Figure([go.Scatter(x=stock_df.index, y=stock_df['High'])])
    fig.update_layout(
        title=stock_df.name+" Stock Price",
        title_x=0.5,
        xaxis_title="Date",
        yaxis_title="Stock Price",
        legend_title="Legend",
        font=dict(
            family="Courier New, monospace",
            size=12,
            color="RebeccaPurple"
        )
    )
    fig.show()

Dropdown=ipw.Dropdown( 
    options=[('Google','GOOG'),('Microsoft Corporation','MSFT'),('Meituan','MPNGF')],description='Stock:') #=[('One', 1), ('Two', 2), ('Three', 3)],

btn=ipw.Button(description='Search',
           disabled=False,
           buttion_style='',
           tooltip='Click me'
)



def findstable(df,date,last,percentage=0.05):
    df=df.loc[date:]
    start=date
    price=df.loc[date,'Open']
    interval=[price*(1-percentage),price*(1+percentage)]
    list1=[]
    day_count=0
    
    for date,row in df.iterrows():
        if (interval[0]<=row['High']<=interval[1]) and (interval[0]<=row['Low']<=interval[1]) and date!=df.index[-1]:
            day_count+=1
        else:
            if day_count>=last or date==df.index[-1]:
                list1.append((price,start,date,interval,day_count))
            start=date    
            price=row['Open']
            interval=[price*(1-percentage),price*(1+percentage)]
            day_count=0
    return pd.DataFrame(list1,columns=['price','Start_date','End_date',str(percentage*100)+'% Interval','day count'])


        
def selectStock(Stock_Search,date):
    date=date.value.strftime("%Y-%m-%d")
    stock = yf.Ticker(Stock_Search.value)
    stock_df=stock.history(period="max")
    stock_df['150moving average']=(stock_df.Open+stock_df.Close).rolling(window=150).mean()/2
    stock_df['50moving average']=(stock_df.Open+stock_df.Close).rolling(window=50).mean()/2
    stock_df=stock_df.loc[date:]    
    stock_df.name=Stock_Search.label
    plot_chart(stock_df)
    #linechart(stock_df)
    df=findstable(stock_df,date,15)
    display(df)
    
date=ipw.DatePicker(
    description='Start Date',
    disabled=False,
    value=datetime(year=2020,month=1,day=3)
)
search_row=ipw.HBox([Dropdown,date,btn])

def is_onclick(b):
    clear_output(True)
    display(search_row)
    selectStock(Dropdown,date)
    
btn.on_click(is_onclick)
btn.click()

,price,Start_date,End_date,5.0% Interval,day count
0,1479.119995,2020-01-21,2020-02-25,"[1405.163995361328, 1553.075994873047]",23
1,1378.280029,2020-05-11,2020-06-08,"[1309.3660278320312, 1447.1940307617188]",18
2,1498.930054,2020-07-24,2020-08-20,"[1423.9835510253906, 1573.8765563964844]",18
3,1790.900024,2020-11-09,2020-11-30,"[1701.3550231933593, 1880.4450256347657]",13
